In [9]:
import os
import re
from datetime import datetime

import ee
import geemap  # optional for visualization/debugging


#####################################
#    SET ALL PARAMETERS #############
#####################################

PROJECT_DIR = "projects/ee-tymc5571-cbi-module/assets"
GEE_PROJECT = "ee-tymc5571-cbi-module"
CRS = "EPSG:5070"
DRIVE_FOLDER = "GEE_Exports_WUMI2024a_allsev"
YEARS = [2003] #None  # e.g. [2003, 2004, 2005]; leave None to run all missing years
DIR_DERIVED = "C:/Users/tymc5571/dev/cbi-module/data/derived/"

DATASET_NAME = "WELTY"
BANDS_NM = "allsev_newtest"



# Output bands to include in exported images.
# You can trim this list freely.
bandsToExport = [
    "CBI_bc",
    "dnbr", "rbr", "rdnbr", "dndvi", "devi", "dndmi", "dmirbi", "post_nbr", "post_mirbi",
    "CBI_bc_masked_relaxed",
    "CBI_bc_masked_conservative",
    "CBI_bc_masked_rap10"
]

#####################################

ee.Authenticate(
    auth_mode='notebook',  # or 'gcloud' or 'appdefault' in other contexts
    scopes=['https://www.googleapis.com/auth/earthengine',
            'https://www.googleapis.com/auth/devstorage.read_write',
            'https://www.googleapis.com/auth/drive']
    #, quiet=False  # force re-authentication even if cached tokens exist
)
ee.Initialize(project=GEE_PROJECT)


In [10]:

#####################################
#   STUDY AREA / POLYGON REGISTRY   #
#####################################

FOREST_MASKS_COLLECTION = ee.ImageCollection(
    "projects/ee-tymc5571-cbi-module/assets/forest_masks"
)

STATES = ee.FeatureCollection("TIGER/2018/States")

WESTERN_STATES = [
    "WA", "OR", "CA", "ID", "NV", "MT", "WY", "UT", "CO", "AZ", "NM"
]

outputRegion = STATES.filter(
    ee.Filter.inList("STUSPS", WESTERN_STATES)
)

# Seasonal state groups from Parks et al.
NORMAL_GROW_STATE_NAMES = [
    "Washington", "Oregon", "California", "Idaho", "Montana",
    "Nevada", "Colorado", "Wyoming", "Utah"
]
SPECIAL_GROW_STATE_NAMES = ["New Mexico", "Arizona"]

studyAreaNormalGrow = STATES.filter(
    ee.Filter.inList("NAME", NORMAL_GROW_STATE_NAMES)
)
studyAreaSpecialGrow = STATES.filter(
    ee.Filter.inList("NAME", SPECIAL_GROW_STATE_NAMES)
)

normalGrowGeom = studyAreaNormalGrow.geometry()
specialGrowGeom = studyAreaSpecialGrow.geometry()

START_DAY_NORMAL = ee.String("152")
END_DAY_NORMAL = ee.String("258")
START_DAY_SPECIAL = ee.String("91")
END_DAY_SPECIAL = ee.String("181")

# Polygon sets
WUMI2024a_c = ee.FeatureCollection(
    f"{PROJECT_DIR}/WUMI2024a_main_fires_unified_simple_with_circles"
)
WUMI2024a = ee.FeatureCollection(
    f"{PROJECT_DIR}/WUMI2024a_main_fires_unified_simple_no_circles"
)
FIREDv2 = ee.FeatureCollection(
    f"{PROJECT_DIR}/firedv2_west_2000_to_2025_events"
)
WELTY = ee.FeatureCollection(
    f"{PROJECT_DIR}/welty_wildfire_west_1984_2020"
)

DATASET_REGISTRY = {
    "WUMI2024a circles": {
        "fc": WUMI2024a_c,
        "asset_slug": "WUMI2024a_c",
        "year_attr": "fire_year",
    },
    "WUMI2024a no-circles": {
        "fc": WUMI2024a,
        "asset_slug": "WUMI2024a",
        "year_attr": "fire_year",
    },
    "FIREDv2": {
        "fc": FIREDv2,
        "asset_slug": "firedv2_west",
        "year_attr": "ig_year",
    },
    "WELTY": {
        "fc": WELTY,
        "asset_slug": "combinedwildland",
        "year_attr": "Fire_Year",
    },
}

firesRaw = DATASET_REGISTRY[DATASET_NAME]["fc"]
fireyear_attribute = DATASET_REGISTRY[DATASET_NAME]["year_attr"]
dataset_nm_use = DATASET_REGISTRY[DATASET_NAME]["asset_slug"]

collection_id = f"{PROJECT_DIR}/{dataset_nm_use}_{BANDS_NM}"



In [11]:

#####################################
#           ASSET HELPERS           #
#####################################

def ensure_imagecollection(collection_id: str) -> None:
    try:
        asset = ee.data.getAsset(collection_id)
        if asset["type"] != "IMAGE_COLLECTION":
            raise ValueError(f"{collection_id} exists but is not an ImageCollection.")
        print(f"ImageCollection already exists: {collection_id}")
    except ee.EEException:
        print(f"Creating ImageCollection: {collection_id}")
        ee.data.createAsset({"type": "IMAGE_COLLECTION"}, collection_id)


def list_assets(parent: str) -> list[str]:
    assets = []

    def _list(parent_id):
        response = ee.data.listAssets({"parent": parent_id})
        for item in response.get("assets", []):
            asset_id = item.get("name")
            asset_type = item.get("type")
            if asset_type in ("FOLDER", "IMAGE_COLLECTION"):
                _list(asset_id)
            else:
                assets.append(asset_id)

    _list(parent)
    return assets


def extract_years_from_assets(asset_list: list[str]) -> list[int]:
    years = []
    pattern = re.compile(r"_(\d{4})$")
    for asset in asset_list:
        match = pattern.search(asset)
        if match:
            years.append(int(match.group(1)))
    return sorted(set(years))


ensure_imagecollection(collection_id)


#####################################
#         YEAR SELECTION            #
#####################################

stats = firesRaw.reduceColumns(
    reducer=ee.Reducer.minMax(),
    selectors=[fireyear_attribute]
)

min_year = int(stats.get("min").getInfo())
max_year = int(stats.get("max").getInfo())

print("Min fire year:", min_year)
print("Max fire year:", max_year)

start_year = max(min_year, 1985)
now = datetime.now()
end_year = now.year - 1 if now.month >= 10 else now.year - 2
end_year = min(end_year, max_year)

viable_years = list(range(start_year, end_year + 1))
print("Viable Landsat years:", viable_years)

asset_string = f"{dataset_nm_use}_{BANDS_NM}"
assets = list_assets(PROJECT_DIR)
filtered_assets = [a for a in assets if asset_string in a]
available_years = extract_years_from_assets(filtered_assets)

years_missing_assets = [y for y in viable_years if y not in available_years]

print("Landsat years with pre-existing assets:", available_years)
print("Landsat years missing assets:", years_missing_assets)

if YEARS is None:
    years_to_process = years_missing_assets
else:
    years_to_process = [int(y) for y in YEARS]

print("Final years to process:", years_to_process)


#####################################
#      RANDOM FOREST COMPONENTS     #
#####################################

rf_bands = ["def", "lat", "rbr", "dmirbi", "dndvi", "post_mirbi"]

cbi = ee.FeatureCollection("users/grumpyunclesean/CBI_predictions/data_for_ee_model")
def_img = ee.Image("users/grumpyunclesean/CBI_predictions/def").rename("def").toInt()
lat = ee.Image.pixelLonLat().select("latitude").rename("lat").round().toInt()

nrow_training_fold = cbi.size()
minLeafPopulation = nrow_training_fold.divide(75).divide(6).round()

fsev_classifier = ee.Classifier.smileRandomForest(
    numberOfTrees=500,
    minLeafPopulation=minLeafPopulation,
    seed=123
).train(cbi, "CBI", rf_bands).setOutputMode("REGRESSION")


#####################################
#          LANDSAT SETUP            #
#####################################

bandList = [
    "dnbr", "rbr", "rdnbr", "dndvi", "devi", "dndmi", "dmirbi",
    "post_nbr", "post_mirbi", "CBI", "CBI_bc"
]

ALL_EXPORTABLE_BANDS = bandList + [
    "CBI_bc_masked_relaxed",
    "CBI_bc_masked_conservative",
    "CBI_bc_masked_rap10"
]

ls9SR = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
ls8SR = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
ls7SR = ee.ImageCollection("LANDSAT/LE07/C02/T1_L2")
ls5SR = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2")
ls4SR = ee.ImageCollection("LANDSAT/LT04/C02/T1_L2")

print("LS Projection", ls9SR.first().projection().getInfo())


def applyScaleFactors(lsImage):
    opticalBands = lsImage.select(["SR_B."]).multiply(0.0000275).add(-0.2)
    return lsImage.addBands(opticalBands, None, True)


def ls8_9_Indices(lsImage):
    nbr = lsImage.normalizedDifference(["SR_B5", "SR_B7"]).toFloat()
    ndvi = lsImage.normalizedDifference(["SR_B5", "SR_B4"]).toFloat()
    ndmi = lsImage.normalizedDifference(["SR_B5", "SR_B6"]).toFloat()
    evi = lsImage.expression(
        "2.5 * ((SR_B5 - SR_B4) / (SR_B5 + 6 * SR_B4 - 7.5 * SR_B2 + 1))",
        {
            "SR_B5": lsImage.select("SR_B5"),
            "SR_B4": lsImage.select("SR_B4"),
            "SR_B2": lsImage.select("SR_B2"),
        }
    ).toFloat()
    mirbi = lsImage.expression(
        "((10 * SR_B6) - (9.8 * SR_B7) + 2)",
        {
            "SR_B6": lsImage.select("SR_B6"),
            "SR_B7": lsImage.select("SR_B7"),
        }
    ).toFloat()
    qa = lsImage.select(["QA_PIXEL"])

    return (
        nbr.addBands([ndvi, ndmi, evi, mirbi, qa])
        .select([0, 1, 2, 3, 4, 5], ["nbr", "ndvi", "ndmi", "evi", "mirbi", "QA_PIXEL"])
        .copyProperties(lsImage, ["system:time_start"])
    )


def ls4_7_Indices(lsImage):
    nbr = lsImage.normalizedDifference(["SR_B4", "SR_B7"]).toFloat()
    ndvi = lsImage.normalizedDifference(["SR_B4", "SR_B3"]).toFloat()
    ndmi = lsImage.normalizedDifference(["SR_B4", "SR_B5"]).toFloat()
    evi = lsImage.expression(
        "2.5 * ((SR_B4 - SR_B3) / (SR_B4 + 6 * SR_B3 - 7.5 * SR_B1 + 1))",
        {
            "SR_B4": lsImage.select("SR_B4"),
            "SR_B3": lsImage.select("SR_B3"),
            "SR_B1": lsImage.select("SR_B1"),
        }
    ).toFloat()
    mirbi = lsImage.expression(
        "((10 * SR_B5) - (9.8 * SR_B7) + 2)",
        {
            "SR_B5": lsImage.select("SR_B5"),
            "SR_B7": lsImage.select("SR_B7"),
        }
    ).toFloat()
    qa = lsImage.select(["QA_PIXEL"])

    return (
        nbr.addBands([ndvi, ndmi, evi, mirbi, qa])
        .select([0, 1, 2, 3, 4, 5], ["nbr", "ndvi", "ndmi", "evi", "mirbi", "QA_PIXEL"])
        .copyProperties(lsImage, ["system:time_start"])
    )


def lsCfmask(lsImage):
    cloudsBitMask = (1 << 3)
    cloudShadowBitMask = (1 << 4)
    snowBitMask = (1 << 5)
    waterBitMask = (1 << 7)

    qa = lsImage.select("QA_PIXEL")
    clear = (
        qa.bitwiseAnd(cloudsBitMask).eq(0)
        .And(qa.bitwiseAnd(cloudShadowBitMask).eq(0))
        .And(qa.bitwiseAnd(snowBitMask).eq(0))
        .And(qa.bitwiseAnd(waterBitMask).eq(0))
    )

    return (
        lsImage.updateMask(clear)
        .select([0, 1, 2, 3, 4])
        .copyProperties(lsImage, ["system:time_start"])
    )


waterMask = ee.Image("UMD/hansen/global_forest_change_2015").select(["datamask"]).eq(1)

ls9 = ls9SR.map(applyScaleFactors).map(ls8_9_Indices).map(lsCfmask)
ls8 = ls8SR.map(applyScaleFactors).map(ls8_9_Indices).map(lsCfmask)
ls7 = ls7SR.map(applyScaleFactors).map(ls4_7_Indices).map(lsCfmask)
ls5 = ls5SR.map(applyScaleFactors).map(ls4_7_Indices).map(lsCfmask)
ls4 = ls4SR.map(applyScaleFactors).map(ls4_7_Indices).map(lsCfmask)

lsCol = ee.ImageCollection(ls9.merge(ls8).merge(ls7).merge(ls5).merge(ls4))


#####################################
#          IMAGE HELPERS            #
#####################################

def fully_masked_image(band_names):
    zeros = ee.List.repeat(0, len(band_names))
    return (
        ee.Image.constant(zeros)
        .rename(band_names)
        .toFloat()
        .updateMask(ee.Image.constant(0))
    )


EMPTY_BASE_IMAGE = fully_masked_image(bandList)
EMPTY_FULL_IMAGE = fully_masked_image(ALL_EXPORTABLE_BANDS)


def specImageryGeom(region_geom, fireYear, startday, endday):
    """
    Compute severity imagery for a single geometry using one seasonal window.
    This is the per-geometry analogue of your earlier specImagery() function.
    """
    region_geom = ee.Geometry(region_geom)
    fireYearDate = ee.Date.parse("YYYY", fireYear)
    startday = ee.Number.parse(startday)
    endday = ee.Number.parse(endday)

    preFireYear = fireYearDate.advance(-1, "year")
    preFireYear2 = fireYearDate.advance(-2, "year")
    postFireYear = fireYearDate.advance(1, "year")

    pre_coll_1 = (
        lsCol.filterBounds(region_geom)
        .filterDate(preFireYear, fireYearDate)
        .filter(ee.Filter.dayOfYear(startday, endday))
    )

    pre_coll_2 = (
        lsCol.filterBounds(region_geom)
        .filterDate(preFireYear2, fireYearDate)
        .filter(ee.Filter.dayOfYear(startday, endday))
    )

    post_coll_1 = (
        lsCol.filterBounds(region_geom)
        .filterDate(postFireYear, fireYearDate.advance(2, "year"))
        .filter(ee.Filter.dayOfYear(startday, endday))
    )

    post_coll_2 = (
        lsCol.filterBounds(region_geom)
        .filterDate(postFireYear, fireYearDate.advance(3, "year"))
        .filter(ee.Filter.dayOfYear(startday, endday))
    )

    preFireIndices = ee.Image(
        ee.Algorithms.If(
            pre_coll_1.size().gt(0),
            pre_coll_1.mean().select(
                [0, 1, 2, 3, 4],
                ["pre_nbr", "pre_ndvi", "pre_ndmi", "pre_evi", "pre_mirbi"]
            ),
            fully_masked_image(["pre_nbr", "pre_ndvi", "pre_ndmi", "pre_evi", "pre_mirbi"])
        )
    )

    preFireIndices2 = ee.Image(
        ee.Algorithms.If(
            pre_coll_2.size().gt(0),
            pre_coll_2.mean().select(
                [0, 1, 2, 3, 4],
                ["pre_nbr", "pre_ndvi", "pre_ndmi", "pre_evi", "pre_mirbi"]
            ),
            fully_masked_image(["pre_nbr", "pre_ndvi", "pre_ndmi", "pre_evi", "pre_mirbi"])
        )
    )

    postFireIndices = ee.Image(
        ee.Algorithms.If(
            post_coll_1.size().gt(0),
            post_coll_1.mean().select(
                [0, 1, 2, 3, 4],
                ["post_nbr", "post_ndvi", "post_ndmi", "post_evi", "post_mirbi"]
            ),
            fully_masked_image(["post_nbr", "post_ndvi", "post_ndmi", "post_evi", "post_mirbi"])
        )
    )

    postFireIndices2 = ee.Image(
        ee.Algorithms.If(
            post_coll_2.size().gt(0),
            post_coll_2.mean().select(
                [0, 1, 2, 3, 4],
                ["post_nbr", "post_ndvi", "post_ndmi", "post_evi", "post_mirbi"]
            ),
            fully_masked_image(["post_nbr", "post_ndvi", "post_ndmi", "post_evi", "post_mirbi"])
        )
    )

    pre_filled = preFireIndices.unmask(preFireIndices2)
    post_filled = postFireIndices.unmask(postFireIndices2)

    fireIndices = pre_filled.addBands(post_filled)

    burnIndices = fireIndices.expression(
        "(b('pre_nbr') - b('post_nbr')) * 1000"
    ).rename("dnbr").toInt().addBands(fireIndices)

    burnIndices2 = burnIndices.expression(
        "b('dnbr') / (b('pre_nbr') + 1.001)"
    ).rename("rbr").toInt().addBands(burnIndices)

    burnIndices3 = burnIndices2.expression(
        "abs(b('pre_nbr')) < 0.001 ? 0.001 : b('pre_nbr')"
    ).abs().sqrt().rename("pre_nbr2").toFloat().addBands(burnIndices2)

    burnIndices4 = burnIndices3.expression(
        "b('dnbr') / b('pre_nbr2')"
    ).rename("rdnbr").toInt().addBands(burnIndices3)

    burnIndices5 = burnIndices4.expression(
        "(b('pre_ndvi') - b('post_ndvi')) * 1000"
    ).rename("dndvi").toInt().addBands(burnIndices4)

    burnIndices6 = burnIndices5.expression(
        "(b('pre_evi') - b('post_evi')) * 1000"
    ).rename("devi").toInt().addBands(burnIndices5)

    burnIndices7 = burnIndices6.expression(
        "(b('pre_ndmi') - b('post_ndmi')) * 1000"
    ).rename("dndmi").toInt().addBands(burnIndices6)

    burnIndices8 = burnIndices7.expression(
        "(b('pre_mirbi') - b('post_mirbi')) * 1000"
    ).rename("dmirbi").toInt().addBands(burnIndices7)

    post_mirbi_1000 = burnIndices8.select("post_mirbi").multiply(1000).toInt()
    burnIndices8 = burnIndices8.addBands(post_mirbi_1000, None, True)

    burnIndices9 = burnIndices8.addBands(def_img)
    burnIndices10 = burnIndices9.addBands(lat)

    cbi_rf = (
        burnIndices10.select(rf_bands)
        .classify(fsev_classifier)
        .rename("CBI")
        .toFloat()
        .multiply(100)
        .floor()
        .divide(100)
    )

    burnIndices11 = cbi_rf.addBands(burnIndices10)

    def bias_correct(bandImage):
        cbi_lo = bandImage.expression("((b('CBI') - 1.5) * 1.3) + 1.5")
        cbi_hi = bandImage.expression("((b('CBI') - 1.5) * 1.175) + 1.5")
        cbi_mg = bandImage.where(bandImage.lte(1.5), cbi_lo).where(bandImage.gt(1.5), cbi_hi)
        return (
            cbi_mg.where(cbi_mg.lt(0), 0)
            .where(cbi_mg.gt(3), 3)
            .multiply(100)
            .floor()
            .divide(100)
            .rename("CBI_bc")
        )

    validMask = (
        pre_filled.select("pre_nbr")
        .add(10)
        .add(post_filled.select("post_nbr"))
        .add(10)
    )

    mask = waterMask.updateMask(validMask)

    burnIndices12 = bias_correct(burnIndices11.select("CBI")).addBands(burnIndices11)
    burnIndices12 = (
        burnIndices12
        .select(bandList, bandList)
        .updateMask(mask)
        .clip(region_geom)
    )

    return burnIndices12


def get_prior_year_forest_mask(fireYear):
    fire_year_num = ee.Number.parse(ee.String(fireYear))
    prior_year = fire_year_num.subtract(1)

    matches = FOREST_MASKS_COLLECTION.filter(ee.Filter.eq("year", prior_year))
    has_match = matches.size().gt(0)

    fully_masked_fallback = (
        ee.Image.constant([1, 1, 1])
        .rename(["relaxed", "conservative", "rap10"])
        .updateMask(ee.Image.constant(0))
    )

    mask_img = ee.Image(
        ee.Algorithms.If(
            has_match,
            matches.first(),
            fully_masked_fallback
        )
    )

    return mask_img.set({
        "mask_year": prior_year,
        "mask_found": has_match
    })


def add_masked_cbi_bands(image, fireYear):
    mask_img = get_prior_year_forest_mask(fireYear)
    cbi_bc = image.select("CBI_bc")

    relaxed_mask = mask_img.select("relaxed").eq(1)
    conservative_mask = mask_img.select("conservative").eq(1)
    rap10_mask = mask_img.select("rap10").eq(1)

    cbi_bc_masked_relaxed = cbi_bc.updateMask(relaxed_mask).rename("CBI_bc_masked_relaxed")
    cbi_bc_masked_conservative = cbi_bc.updateMask(conservative_mask).rename("CBI_bc_masked_conservative")
    cbi_bc_masked_rap10 = cbi_bc.updateMask(rap10_mask).rename("CBI_bc_masked_rap10")

    return image.addBands([
        cbi_bc_masked_relaxed,
        cbi_bc_masked_conservative,
        cbi_bc_masked_rap10
    ])


def compute_single_fire_image(fire_feat, fireYear):
    fire_feat = ee.Feature(fire_feat)
    fire_geom = fire_feat.geometry()

    normal_piece = fire_geom.intersection(normalGrowGeom, ee.ErrorMargin(1))
    special_piece = fire_geom.intersection(specialGrowGeom, ee.ErrorMargin(1))

    normal_img = ee.Image(
        ee.Algorithms.If(
            normal_piece.area(1).gt(0),
            specImageryGeom(normal_piece, fireYear, START_DAY_NORMAL, END_DAY_NORMAL).toFloat(),
            EMPTY_BASE_IMAGE
        )
    ).select(bandList).toFloat()

    special_img = ee.Image(
        ee.Algorithms.If(
            special_piece.area(1).gt(0),
            specImageryGeom(special_piece, fireYear, START_DAY_SPECIAL, END_DAY_SPECIAL).toFloat(),
            EMPTY_BASE_IMAGE
        )
    ).select(bandList).toFloat()

    combined = ee.ImageCollection.fromImages([normal_img, special_img]).mosaic().clip(fire_geom)
    combined = add_masked_cbi_bands(combined, fireYear)
    combined = combined.select(ALL_EXPORTABLE_BANDS).toFloat()

    return combined.set({
        "Fire_Year": fireYear,
        "source_fire_year": fire_feat.get(fireyear_attribute)
    })


def compute_year_image(year_int):
    """
    Compute all fires for a year at per-perimeter scale, mosaic them,
    and return one sparse year image.
    """
    year_int = int(year_int)
    fireYear = str(year_int)

    year_fires = firesRaw.filter(ee.Filter.eq(fireyear_attribute, year_int))
    n_fires = year_fires.size()

    fire_list = year_fires.toList(n_fires)

    per_fire_images = ee.List.sequence(0, n_fires.subtract(1)).map(
        lambda i: compute_single_fire_image(
            ee.Feature(fire_list.get(i)),
            fireYear
        )
    )

    year_img = ee.ImageCollection.fromImages(per_fire_images).mosaic()

    # Final select keeps only requested outputs, but all upstream functionality remains.
    year_img = year_img.select(bandsToExport).toFloat()

    return year_img.set({
        "year": year_int,
        "Fire_Year": fireYear,
        "n_fires": n_fires
    })


#####################################
#             EXPORTS               #
#####################################

def exports(image: ee.Image,
            region_geom: ee.Geometry,
            asset_folder_base: str,
            base_slug: str,
            *,
            suffix: str = "",
            drive_folder: str = "GEE_Exports",
            scale: int = 30,
            crs: str = "EPSG:5070",
            maxPixels: float = 1e13):
    desc = f"{base_slug}{suffix}"
    asset_id = f"{asset_folder_base}/{desc}"

    task_asset = ee.batch.Export.image.toAsset(
        image=image,
        description=desc + "_toAsset",
        assetId=asset_id,
        region=region_geom,
        scale=scale,
        crs=crs,
        maxPixels=maxPixels
    )
    task_asset.start()

    # Uncomment if you also want Drive exports.
    # task_drive = ee.batch.Export.image.toDrive(
    #     image=image,
    #     description=desc + "_toDrive",
    #     folder=drive_folder,
    #     fileNamePrefix=desc,
    #     region=region_geom,
    #     scale=scale,
    #     crs=crs,
    #     maxPixels=maxPixels
    # )
    # task_drive.start()

    return {
        "asset_task": task_asset.id
        # "drive_task": task_drive.id
    }


#####################################
#          RUN THE EXPORTS          #
#####################################

if not years_to_process:
    print("No years to process.")
else:
    for y in years_to_process:
        print(f"Building year {y}...")

        img_y = compute_year_image(y)

        exports(
            image=img_y,
            region_geom=outputRegion.geometry(),  # keep full western US export extent
            asset_folder_base=collection_id,
            base_slug=asset_string,
            suffix=f"_{y}",
            drive_folder=DRIVE_FOLDER,
            scale=30,
            crs=CRS,
            maxPixels=1e13
        )

    print("Started exports for years:", years_to_process)

tasks = ee.data.getTaskList()
for t in tasks:
    print(f"[{t['state']:<10}] {t.get('task_type'):<14} {t.get('description')}  id={t.get('id')}")

ImageCollection already exists: projects/ee-tymc5571-cbi-module/assets/combinedwildland_allsev_newtest
Min fire year: 1984
Max fire year: 2020
Viable Landsat years: [1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]
Landsat years with pre-existing assets: []
Landsat years missing assets: [1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]
Final years to process: [2003]
LS Projection {'type': 'Projection', 'crs': 'EPSG:32628', 'transform': [30, 0, 340185, 0, -30, 8808615]}
Building year 2003...
Started exports for years: [2003]
[READY     ] EXPORT_IMAGE   combinedwildland_allsev_newtest_2003_toAsset  id=5LDCHNA4HENJVNMI6NPX643O
[FAILED    ] EXPORT_IMAGE   combin